# Ручная реализация оптимизаторов 

In [20]:
import numpy as np
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Adagrad

In [54]:
class AdaGradLinearRegression:
    def __init__(self, learning_rate=0.001, epsilon=1e-5):
        self.lr = learning_rate
        self.epsilon = epsilon
        
        self.w = None
        self.b = 0.0
        
        self.G_w = None
        self.G_b = 0.0
        
        self.mean = None
        self.std = None
        
        self.y_mean = None
        self.y_std = None

    def _normalize(self, X):
        if self.mean is None or self.std is None:
            self.mean = np.mean(X, axis=0)
            self.std = np.std(X, axis=0)
            self.std[self.std == 0] = 1e-8
        return (X - self.mean) / self.std

    def _get_gradient(self, X_scaled, y_scaled):
        N = X_scaled.shape[0]
        predictions = np.dot(X_scaled, self.w) + self.b
        err = predictions - y_scaled
        
        grad_w = (2 / N) * np.dot(X_scaled.T, err)
        grad_b = (2 / N) * np.sum(err)
        return grad_w, grad_b

    def __call__(self, X, y, epochs=500, verbose=True):
        X_scaled = self._normalize(X)
        if self.y_mean is None or self.y_std is None:
            self.y_mean = np.mean(y)
            self.y_std = np.std(y)
            if self.y_std == 0: 
                self.y_std = 1e-8
        y_scaled = (y - self.y_mean) / self.y_std
        
        num_features = X.shape[1]
        if self.w is None:
            self.w = np.zeros(num_features)
            self.G_w = np.zeros(num_features)
            self.G_b = 0.0

        for epoch in range(1, epochs + 1):
            grad_w, grad_b = self._get_gradient(X_scaled, y_scaled)
            
            self.G_w += grad_w ** 2
            self.G_b += grad_b ** 2
            
            self.w -= (self.lr / (np.sqrt(self.G_w) + self.epsilon)) * grad_w
            self.b -= (self.lr / (np.sqrt(self.G_b) + self.epsilon)) * grad_b
            
            if verbose and (epoch == 1 or epoch % 50 == 0):
                current_preds_scaled = np.dot(X_scaled, self.w) + self.b
                current_preds_original = current_preds_scaled * self.y_std + self.y_mean
                mse = np.mean((current_preds_original - y) ** 2)
                print(f"Эпоха {epoch:3d} | Текущая ошибка (MSE): {mse:.4f}")
                
        return self.w, self.b

    def predict(self, X):
        if self.w is None:
            raise ValueError("Модель еще не обучена")
        X_scaled = (X - self.mean) / self.std
        preds_scaled = np.dot(X_scaled, self.w) + self.b
        return preds_scaled * self.y_std + self.y_mean

In [55]:
np.random.seed(69)
X_random = np.random.rand(200, 19) * 100

true_w = np.random.uniform(-5, 5, size=19)
true_b = 15.0
noise = np.random.normal(0, 2, size=200)
y_random = np.dot(X_random, true_w) + true_b + noise

adagrad_model = AdaGradLinearRegression(learning_rate=0.4, epsilon=1e-8)

print("~~~~~~~ Старт обучения ~~~~~~~~")
final_w, final_b = adagrad_model(X_random, y_random, epochs=500)
print("~~~~~~~ Обучение завершено ~~~~~~~\n")

original_w_found = (adagrad_model.w * adagrad_model.y_std) / adagrad_model.std
original_b_found = (adagrad_model.b * adagrad_model.y_std) + adagrad_model.y_mean - np.sum((adagrad_model.mean * adagrad_model.w * adagrad_model.y_std) / adagrad_model.std)

print(f"Истинный b: {true_b}")
print(f"Найденный денормализованный b: {original_b_found:.4f}")
print(f"Максимальное отклонение по весам W: {np.max(np.abs(true_w - original_w_found)):.4f}")

~~~~~~~ Старт обучения ~~~~~~~~
Эпоха   1 | Текущая ошибка (MSE): 190525.2738
Эпоха  50 | Текущая ошибка (MSE): 3.0592
Эпоха 100 | Текущая ошибка (MSE): 3.0562
Эпоха 150 | Текущая ошибка (MSE): 3.0562
Эпоха 200 | Текущая ошибка (MSE): 3.0562
Эпоха 250 | Текущая ошибка (MSE): 3.0562
Эпоха 300 | Текущая ошибка (MSE): 3.0562
Эпоха 350 | Текущая ошибка (MSE): 3.0562
Эпоха 400 | Текущая ошибка (MSE): 3.0562
Эпоха 450 | Текущая ошибка (MSE): 3.0562
Эпоха 500 | Текущая ошибка (MSE): 3.0562
~~~~~~~ Обучение завершено ~~~~~~~

Истинный b: 15.0
Найденный денормализованный b: 14.9046
Максимальное отклонение по весам W: 0.0118


## AdamW

In [47]:
class AdamWLinearRegression:
    def __init__(self, learning_rate=0.05, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=0.01):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.weight_decay = weight_decay
        
        self.w = None
        self.b = 0.0
        # Первые моменты (m) и вторые моменты (v)
        self.m_w, self.v_w = None, None
        self.m_b, self.v_b = 0.0, 0.0
        # счетчик шагов (нужен для исправления смещения beta^t)
        self.t = 0
        # нормализация
        self.mean = None
        self.std = None

    def _normalize(self, X):
        if self.mean is None or self.std is None:
            self.mean = np.mean(X, axis=0)
            self.std = np.std(X, axis=0)
            self.std[self.std == 0] = 1e-8
        return (X - self.mean) / self.std

    def _get_gradient(self, X_scaled, y):
        N = X_scaled.shape[0]
        predictions = np.dot(X_scaled, self.w) + self.b
        err = predictions - y
        # Чистые градиенты без L2-регуляризации (в AdamW она отделена!)
        grad_w = (2 / N) * np.dot(X_scaled.T, err)
        grad_b = (2 / N) * np.sum(err)
        return grad_w, grad_b

    def __call__(self, X, y, epochs=500, verbose=True):
        X_scaled = self._normalize(X)
        num_features = X.shape[1]
        # инициализация матриц при первом вызове
        if self.w is None:
            self.w = np.zeros(num_features)
            self.m_w = np.zeros(num_features)
            self.v_w = np.zeros(num_features)
            
        for epoch in range(1, epochs + 1):
            self.t += 1  # Увеличиваем шаг для формулы коррекции
            grad_w, grad_b = self._get_gradient(X_scaled, y)
            # шаг для весов (w) 
            self.m_w = self.beta1 * self.m_w + (1 - self.beta1) * grad_w
            self.v_w = self.beta2 * self.v_w + (1 - self.beta2) * (grad_w ** 2)
            
            # так называемый bias correction
            m_w_corrected = self.m_w / (1 - self.beta1 ** self.t)
            v_w_corrected = self.v_w / (1 - self.beta2 ** self.t)
            
            self.w -= self.lr * self.weight_decay * self.w + (self.lr / (np.sqrt(v_w_corrected) + self.epsilon)) * m_w_corrected
            
            # шаг AdamW для b (без weight decay)
            self.m_b = self.beta1 * self.m_b + (1 - self.beta1) * grad_b
            self.v_b = self.beta2 * self.v_b + (1 - self.beta2) * (grad_b ** 2)
            m_b_corrected = self.m_b / (1 - self.beta1 ** self.t)
            b_v_corrected = self.v_b / (1 - self.beta2 ** self.t)
            self.b -= (self.lr / (np.sqrt(b_v_corrected) + self.epsilon)) * m_b_corrected
            # Вывод логов
            if verbose and (epoch == 1 or epoch % 50 == 0):
                current_preds = np.dot(X_scaled, self.w) + self.b
                mse = np.mean((current_preds - y) ** 2)
                print(f"Эпоха {epoch:3d} | MSE: {mse:.4f}")
        return self.w, self.b

    def predict(self, X):
        if self.w is None:
            raise ValueError("Нет ничего, что можно было бы предсказать")
        X_scaled = self._normalize(X)
        return np.dot(X_scaled, self.w) + self.b

In [65]:
np.random.seed(69)
X_rand = np.random.randn(300, 19) * 50

true_weights = np.random.uniform(-3, 3, size=19)
true_bias = 42.0
y_rand = np.dot(X_rand, true_weights) + true_bias + np.random.normal(0, 5, size=300)

adamw_model = AdamWLinearRegression(learning_rate=1.0, weight_decay=0.0001)
print("~~~~~ Старт обучения AdamW ~~~~~~~")
w_found, b_found = adamw_model(X_rand, y_rand, epochs=500)
print("~~~~~ Обучение завершено ~~~~~~~~\n")

print(f"Оригинальный b: {true_bias}")
print(f"Найденный (нормализованный) b: {b_found:.4f}") 

# оригинальные веса W
original_w = adamw_model.w / adamw_model.std
# Восстанавливаем оригинальный bias b
original_b = b_found - np.sum((adamw_model.mean * adamw_model.w) / adamw_model.std)
print(f"Денормализованный (настоящий) b: {original_b:.4f}")
print(f"Максимальное отклонение по весам W: {np.max(np.abs(true_weights - original_w)):.4f}")

~~~~~ Старт обучения AdamW ~~~~~~~
Эпоха   1 | MSE: 149747.7807
Эпоха  50 | MSE: 55590.1801
Эпоха 100 | MSE: 17980.2174
Эпоха 150 | MSE: 4991.7540
Эпоха 200 | MSE: 1218.8743
Эпоха 250 | MSE: 284.9345
Эпоха 300 | MSE: 80.2425
Эпоха 350 | MSE: 37.7769
Эпоха 400 | MSE: 28.5498
Эпоха 450 | MSE: 26.1831
Эпоха 500 | MSE: 25.3785
~~~~~ Обучение завершено ~~~~~~~~

Оригинальный b: 42.0
Найденный (нормализованный) b: 24.8757
Денормализованный (настоящий) b: 41.9773
Максимальное отклонение по весам W: 0.0192


## Сэмплированный датасет

In [66]:
CONFIG = {
    "EPOCHS": 500,                                          
    "SEED": 67,
}

In [67]:
# Находим нужный сэмплированный датасет
target_relative_path = Path("..") / "Dataset (Farhat)" / "dataset_sample_3000.csv"
dataset_path = target_relative_path.resolve()

if not dataset_path.exists():
    raise FileNotFoundError(f"ОШИБКА: файл датасета не найден по пути: {dataset_path}")
print(f"Успешно обнаружен файл датасета по пути: {dataset_path}")

Успешно обнаружен файл датасета по пути: /Users/test/Desktop/Various_Linear_Regression_Factors/Dataset (Farhat)/dataset_sample_3000.csv


In [68]:
df2 = pd.read_csv(dataset_path)
X_raw = df2.drop(columns=['Log_Цена']).values
y_raw = df2['Log_Цена'].values 

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=CONFIG["SEED"]
)
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_raw.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test_raw.reshape(-1, 1)).ravel()

In [69]:
modelAdamW = AdamWLinearRegression(learning_rate=0.01, weight_decay=0.0001)

print("~~~~~~ Обучение на Train ~~~~~~")
modelAdamW(X_train_raw, y_train_scaled, epochs=1000)

y_test_pred_scaled = modelAdamW.predict(X_test_raw)
y_test_pred_log = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

test_mse = np.mean((y_test_pred_log - y_test_raw) ** 2)
print(f"\nФинальный MSE на тестовой выборке: {test_mse:.5f}")

~~~~~~ Обучение на Train ~~~~~~
Эпоха   1 | MSE: 0.9306
Эпоха  50 | MSE: 0.2432
Эпоха 100 | MSE: 0.1699
Эпоха 150 | MSE: 0.1649
Эпоха 200 | MSE: 0.1646
Эпоха 250 | MSE: 0.1645
Эпоха 300 | MSE: 0.1645
Эпоха 350 | MSE: 0.1645
Эпоха 400 | MSE: 0.1645
Эпоха 450 | MSE: 0.1645
Эпоха 500 | MSE: 0.1645
Эпоха 550 | MSE: 0.1645
Эпоха 600 | MSE: 0.1645
Эпоха 650 | MSE: 0.1645
Эпоха 700 | MSE: 0.1645
Эпоха 750 | MSE: 0.1645
Эпоха 800 | MSE: 0.1645
Эпоха 850 | MSE: 0.1645
Эпоха 900 | MSE: 0.1645
Эпоха 950 | MSE: 0.1645
Эпоха 1000 | MSE: 0.1645

Финальный MSE на тестовой выборке: 0.04065


In [70]:
modelAdaGrad = AdaGradLinearRegression(learning_rate=0.1, epsilon=1e-8)

print("--- Старт обучения AdaGrad на Train ---")
modelAdaGrad(X_train_raw, y_train_scaled, epochs=1000, verbose=True)

y_test_pred_scaled = modelAdaGrad.predict(X_test_raw)
y_test_pred_log = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

test_mse = np.mean((y_test_pred_log - y_test_raw) ** 2)
print(f"\nФинальный MSE (AdaGrad) на тестовой выборке: {test_mse:.5f}")

--- Старт обучения AdaGrad на Train ---
Эпоха   1 | Текущая ошибка (MSE): 0.6296
Эпоха  50 | Текущая ошибка (MSE): 0.1664
Эпоха 100 | Текущая ошибка (MSE): 0.1646
Эпоха 150 | Текущая ошибка (MSE): 0.1645
Эпоха 200 | Текущая ошибка (MSE): 0.1645
Эпоха 250 | Текущая ошибка (MSE): 0.1645
Эпоха 300 | Текущая ошибка (MSE): 0.1645
Эпоха 350 | Текущая ошибка (MSE): 0.1645
Эпоха 400 | Текущая ошибка (MSE): 0.1645
Эпоха 450 | Текущая ошибка (MSE): 0.1645
Эпоха 500 | Текущая ошибка (MSE): 0.1645
Эпоха 550 | Текущая ошибка (MSE): 0.1645
Эпоха 600 | Текущая ошибка (MSE): 0.1645
Эпоха 650 | Текущая ошибка (MSE): 0.1645
Эпоха 700 | Текущая ошибка (MSE): 0.1645
Эпоха 750 | Текущая ошибка (MSE): 0.1645
Эпоха 800 | Текущая ошибка (MSE): 0.1645
Эпоха 850 | Текущая ошибка (MSE): 0.1645
Эпоха 900 | Текущая ошибка (MSE): 0.1645
Эпоха 950 | Текущая ошибка (MSE): 0.1645
Эпоха 1000 | Текущая ошибка (MSE): 0.1645

Финальный MSE (AdaGrad) на тестовой выборке: 0.04065
